# Аналитика поверх трёх пайплайнов


Читает историю с начала месяца по текущий день из рабочих таблиц,
считает нагрузку, качество и работу операторов, выкладывает
результат в отдельную таблицу аналитики.

Рабочие ноутбуки (monitoring / alpha_registry / omega_registry) не трогает — только читает
то, что они уже выгрузили.

Порядок работы в DataSpell:
    1. Ячейка 1 — настройки
    2. Ячейка 2 — сбор истории (долгая, но кэшируется)
    3. Ячейка 3 — расчёты, проверка цифр на экране
    4. Ячейка 4 — выгрузка в таблицу аналитики


## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from etl_core import (
    DATA_DIR, CREDENTIALS, env,
    DateResolver, HistoryCache, SheetsLoader, Serializer,
    Palette, month_title, preview, log, RU_MONTHS, HOURS, TIME_LABELS,
)

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# etl_core обновляется вместе с ноутбуками, а Python держит уже
# загруженный модуль в памяти: заменить файл мало, нужен Restart Kernel.
# Проверяем сразу, иначе это выяснится на середине выгрузки.
_need = ["archive_to_excel", "reset_sheet_format", "prune_sheets",
         "reorder_sheets", "req_number_format", "req_sign_colors",
         "req_color_scale", "batch_get", "list_date_sheets"]
_missing = [m for m in _need if not hasattr(SheetsLoader, m)]
if _missing:
    raise ImportError(
        "etl_core.py устарел — в нём нет методов: " + ", ".join(_missing)
        + "\n\n1. Положите свежий etl_core.py в папку etl/"
        + "\n2. Нажмите Restart Kernel — без этого Python продолжит "
          "использовать старую версию из памяти"
    )

CACHE_DIR = Path(BASE_DIR) / "cache"

# При смене месяца прошлый месяц уходит в Excel — до того, как таблица
# будет переписана. Файл ложится в папку data/.
ARCHIVE_PREVIOUS = True
ARCHIVE_DIR = Path(BASE_DIR)      # папка data/

# Таблица аналитики ОДНА и та же всегда — открывается по ID, поэтому
# её можно как угодно переименовывать и переставлять, ссылка и закладка
# в браузере остаются рабочими.
#
# Аналитика — производные данные: её всегда можно пересчитать из
# Alpha, Omega и «Мониторинга». Поэтому хранить историю в Google
# незачем: при смене месяца прошлый месяц сохраняется в Excel рядом
# со скриптами, а таблица переписывается под новый.
ANALYTICS_ID = env("ANALYTICS_SHEET_ID")

# Откуда читать историю
SRC_PROJECTS = {
    "kind": "projects",
    "id": env("MONITORING_SHEET_ID"),
    "title": None,                       # таблица не месячная — по ID
}
SRC_ALPHA = {
    "kind": "alpha",
    "id": env("ALPHA_SHEET_ID"),
    "title": "Alpha {month} {year}",
}
SRC_OMEGA = {
    "kind": "omega",
    "id": env("OMEGA_SHEET_ID"),
    "title": "Omega {month} {year}",
}
SOURCES = [SRC_PROJECTS, SRC_ALPHA, SRC_OMEGA]

# Alpha и Omega остаются МЕСЯЧНЫМИ таблицами, поэтому они ищутся
# по названию с подстановкой месяца. Это про источники, а не про
# саму аналитику: у неё таблица одна и открывается по ID.
SOURCE_BY_TITLE = True

# Отчётная дата. None — сегодняшняя; можно задать вручную: date(2026, 7, 26)
REPORT_DATE = None

# Читать реестры CRM ради средней длительности разговора.
# Это самая тяжёлая часть: сотни строк на каждый день. Выключите, если долго.
LOAD_DURATIONS = True

PROJECT_ORDER = ["Beta", "Gamma", "Delta", "Epsilon", "Zeta"]


cache_hourly = HistoryCache(CACHE_DIR / "hourly.csv")
cache_ops = HistoryCache(CACHE_DIR / "operators.csv")
# Уровень обслуживания АТС считает сам, но в Google Sheets он не
# попадает — его складывает в кэш monitoring.ipynb при ежедневной выгрузке
cache_sl = HistoryCache(CACHE_DIR / "service_level.csv")

## ЯЧЕЙКА 2 — СБОР ИСТОРИИ

In [ ]:
def _num(v) -> float:
    """Строка из Sheets -> число. Пустое и мусор -> 0."""
    if v is None or v == "":
        return 0.0
    s = str(v).replace("\xa0", "").replace(" ", "").replace(",", ".").replace("%", "")
    try:
        return float(s)
    except ValueError:
        return 0.0


def _hour_from_label(v) -> "int | None":
    """'00:00-01:00' -> 0. Не разобралось -> None."""
    m = re.match(r"^\s*(\d{1,2})\s*[:.]", str(v))
    return int(m.group(1)) if m else None


def _hour_from_dt(v) -> "int | None":
    """'2026-07-26 09:04:39' -> 9."""
    m = re.search(r"(?:^|\s)(\d{1,2}):\d{2}", str(v))
    return int(m.group(1)) if m else None


def _norm_operator(name: str) -> str:
    """Operator1 / operator3 / «Оператор 1» -> «Оператор 1»."""
    m = re.search(r"(\d+)", str(name))
    return f"Оператор {m.group(1)}" if m else str(name).strip()


def _index_by_header(header: list, wanted: dict) -> dict:
    """
    Находит позиции колонок ПО НАЗВАНИЮ, а не по номеру.

    Нужно потому, что в старых листах Alpha между «Все вызовы»
    и «Принятые входящие» сидит фантомный столбец «#», а в новых его
    нет. По позиции данные за разные дни молча съехали бы на колонку.
    """
    norm = [str(h).strip().lower() for h in header]
    out = {}
    for key, title in wanted.items():
        t = title.strip().lower()
        for i, h in enumerate(norm):
            if h == t:
                out[key] = i
                break
    return out


# ---------- парсеры листов ----------

def parse_projects(values: list, d: date) -> pd.DataFrame:
    """
    Лист «Мониторинга показателей», диапазон A1:G400.

    Колонка «Часы» — объединённые ячейки, значение приходит только
    в первой строке каждого часа. Остальные дотягиваются вперёд.
    """
    if len(values) < 2:
        return pd.DataFrame()

    rows, cur_hour = [], None
    for r in values[1:]:
        r = list(r) + [""] * (7 - len(r))
        h = _hour_from_label(r[0])
        if h is not None:
            cur_hour = h
        project = str(r[1]).strip()
        if cur_hour is None or not project or project.lower().startswith("итог"):
            continue
        rows.append({
            "Дата": d, "Источник": "Проекты", "Проект": project, "Час": cur_hour,
            "IVR": _num(r[2]), "Входящие": _num(r[3]),
            "Потерянные": _num(r[4]), "Исходящие": _num(r[5]),
        })
    return pd.DataFrame(rows)


def _find_header_row(grid: list, marker: str) -> "int | None":
    """
    Ищет строку-шапку по характерному слову в ней.

    Блоки на листах Alpha стоят не на фиксированных ячейках:
    старые листы собирались вручную, и сводка съезжала то на строку
    выше, то правее. Поэтому шапка ищется по тексту, а не по адресу.
    """
    m = marker.strip().lower()
    for i, row in enumerate(grid):
        if any(m in str(c).strip().lower() for c in row):
            return i
    return None


def _describe_grid(grid: list) -> str:
    """
    Коротко описывает, что нашлось в прочитанном куске листа.

    Нужно, когда блок не обнаружен: по описанию видно, пуста ли
    область вообще (тогда блока на листе нет) или там что-то есть,
    но названо иначе.
    """
    cells = [str(c).strip() for row in grid for c in row if str(c).strip()]
    if not cells:
        return "Область справа от реестра пустая — блока на листе нет"
    sample = ", ".join(list(dict.fromkeys(cells))[:8])
    return f"Непустых ячеек: {len(cells)}. Что там: {sample}"


def _find_hour_col(rows: list) -> int:
    """Находит колонку с подписями часов вида «09:00-10:00»."""
    for row in rows:
        for j, c in enumerate(row):
            if re.match(r"^\s*\d{1,2}:\d{2}\s*-\s*\d{1,2}:\d{2}", str(c)):
                return j
    return 0


def parse_alpha_hourly(grid: list, d: date) -> pd.DataFrame:
    """Почасовая сводка Alpha. Блок ищется по шапке в пределах листа."""
    hdr = _find_header_row(grid, "Принятые входящие")
    if hdr is None:
        log.warning("Alpha %s: блок почасовой сводки не найден. %s",
                    d, _describe_grid(grid))
        return pd.DataFrame()

    idx = _index_by_header(grid[hdr], {
        "in": "Принятые входящие вызовы",
        "lost": "Пропущенные вызовы",
        "out": "Исходящие вызовы",
    })
    if len(idx) < 3:
        log.warning("Alpha %s: в шапке не хватает колонок (%s)",
                    d, grid[hdr])
        return pd.DataFrame()

    body = grid[hdr + 1:]
    hcol = _find_hour_col(body)

    rows = []
    for r in body:
        h = _hour_from_label(r[hcol]) if len(r) > hcol else None
        if h is None:
            continue                      # «Итого», пустые строки, чужие блоки
        g = lambda k: _num(r[idx[k]]) if idx[k] < len(r) else 0.0
        rows.append({
            "Дата": d, "Источник": "Alpha", "Проект": "Alpha", "Час": h,
            "IVR": 0.0, "Входящие": g("in"),
            "Потерянные": g("lost"), "Исходящие": g("out"),
        })
        if len(rows) >= 24:
            break                         # дальше начнётся следующий блок
    return pd.DataFrame(rows)


def parse_alpha_operators(grid: list, d: date) -> pd.DataFrame:
    """Разбивка по операторам Alpha. Блок тоже ищется по шапке."""
    hdr = None
    for i, row in enumerate(grid):
        if any(re.match(r"^операторы?\s*\d+", str(c).strip().lower())
               for c in row):
            hdr = i
            break
    if hdr is None:
        log.warning("Alpha %s: блок операторов не найден", d)
        return pd.DataFrame()

    header = [str(h).strip() for h in grid[hdr]]
    ops = {i: h for i, h in enumerate(header)
           if re.match(r"^операторы?\s*\d+", h.lower())}

    body = grid[hdr + 1:]
    hcol = _find_hour_col(body)

    rows, seen = [], 0
    for r in body:
        if len(r) <= hcol or _hour_from_label(r[hcol]) is None:
            continue                      # «Итого» не берём, посчитаем сами
        for i, name in ops.items():
            if i < len(r):
                rows.append({
                    "Дата": d, "Источник": "Alpha",
                    "Оператор": _norm_operator(name),
                    "Принято": _num(r[i]), "Секунды": np.nan, "Звонков": 0,
                })
        seen += 1
        if seen >= 24:
            break
    return pd.DataFrame(rows)


def parse_crm_registry(values: list, d: date, source: str,
                       col_dir: int, col_dur: int, col_op: int) -> tuple:
    """
    Реестр CRM с листа. Возвращает (почасовая сводка, операторы).

    Для Omega это единственный источник — АТС там нет, поэтому
    потерянных звонков в данных не существует.
    Для Alpha отсюда берётся только длительность разговоров.
    """
    if len(values) < 2:
        return pd.DataFrame(), pd.DataFrame()

    hourly, ops = [], []
    for r in values[1:]:
        if not r or len(r) <= max(col_dir, col_dur, col_op):
            continue
        direction = str(r[col_dir]).strip().lower()
        h = _hour_from_dt(r[3]) if len(r) > 3 else None
        if h is None:
            continue
        inbound = direction.startswith("вход")
        hourly.append({"Час": h, "Входящие": int(inbound),
                       "Исходящие": int(not inbound)})
        if inbound:
            ops.append({"Оператор": _norm_operator(r[col_op]),
                        "Секунды": _num(r[col_dur])})

    hdf = pd.DataFrame()
    if hourly:
        g = pd.DataFrame(hourly).groupby("Час", as_index=False).sum()
        hdf = pd.DataFrame({
            "Дата": d, "Источник": source, "Проект": source,
            "Час": g["Час"], "IVR": 0.0, "Входящие": g["Входящие"].astype(float),
            "Потерянные": np.nan, "Исходящие": g["Исходящие"].astype(float),
        })

    odf = pd.DataFrame()
    if ops:
        g = pd.DataFrame(ops).groupby("Оператор", as_index=False).agg(
            Принято=("Секунды", "size"), Секунды=("Секунды", "sum"))
        g["Звонков"] = g["Принято"]
        odf = g.assign(Дата=d, Источник=source)

    return hdf, odf


# ---------- сбор ----------

def fetch_source(src: dict, d_from: date, d_to: date,
                 known: set) -> tuple:
    """
    Читает недостающие дни одного источника.

    Уже закэшированные даты пропускаются — из Sheets тянутся
    только новые. Возвращает (почасовые данные, операторы, отчёт).
    """
    loader = SheetsLoader(CREDENTIALS, src["id"])
    if src["title"] and SOURCE_BY_TITLE:
        loader.connect(title=month_title(src["title"], d_to))
    else:
        loader.connect()

    sheets = loader.list_date_sheets(d_from, d_to)
    need = {d: t for d, t in sheets.items() if d not in known}

    report = {"таблица": loader.sh.title, "листов найдено": len(sheets),
              "уже в кэше": len(sheets) - len(need), "читаю": len(need)}
    if not need:
        return pd.DataFrame(), pd.DataFrame(), report

    q = loader.quote_sheet
    hourly_all, ops_all = [], []

    if src["kind"] == "projects":
        # Высота сетки зависит от числа проектов: 6 x 24 часа = 144 строки
        # данных. Читаем с запасом, до высоты листа (SHEET_ROWS в
        # monitoring.ipynb): пустые строки и «Итого» парсер пропускает.
        ranges = [f"{q(t)}!A1:G400" for t in need.values()]
        for d, vals in zip(need, _chunked_get(loader, ranges)):
            hourly_all.append(parse_projects(vals, d))

    elif src["kind"] == "alpha":
        # Один широкий кусок на день, и из него разбираются ОБА блока —
        # почасовая сводка и операторы. Просить один и тот же диапазон
        # дважды нельзя: API возвращает по одному ответу на уникальный
        # диапазон, ответов приходит меньше, чем запросов, и разбор
        # съезжает на чужие данные. Именно так пропадали операторы.
        ranges, tags = [], []
        for d, t in need.items():
            ranges.append(f"{q(t)}!K1:AD200")
            tags.append((d, "blocks"))
            if LOAD_DURATIONS:
                ranges.append(f"{q(t)}!A1:J900")
                tags.append((d, "crm"))

        got = _chunked_get(loader, ranges)
        if len(got) != len(ranges):
            raise RuntimeError(
                f"Запрошено диапазонов {len(ranges)}, получено {len(got)}. "
                f"Разбор пришлось бы делать вслепую — останавливаюсь."
            )

        for (d, tag), vals in zip(tags, got):
            if tag == "blocks":
                hourly_all.append(parse_alpha_hourly(vals, d))
                ops_all.append(parse_alpha_operators(vals, d))
            else:
                _, odf = parse_crm_registry(vals, d, "Alpha", 1, 5, 7)
                if not odf.empty:
                    ops_all.append(odf.assign(Принято=0))   # принято уже есть

    elif src["kind"] == "omega":
        ranges = [f"{q(t)}!A1:M900" for t in need.values()]
        for d, vals in zip(need, _chunked_get(loader, ranges)):
            hdf, odf = parse_crm_registry(vals, d, "Omega", 1, 5, 9)
            hourly_all.append(hdf)
            ops_all.append(odf)

    cat = lambda lst: (pd.concat([x for x in lst if not x.empty],
                                 ignore_index=True)
                       if any(not x.empty for x in lst) else pd.DataFrame())
    return cat(hourly_all), cat(ops_all), report


def _chunked_get(loader: SheetsLoader, ranges: list, size: int = 25) -> list:
    """Разбивает пакетное чтение на порции — длинный запрос API не любит."""
    out = []
    for i in range(0, len(ranges), size):
        out += loader.batch_get(ranges[i:i + size])
    return out


def collect_history(d_from: "date | None" = None,
                    d_to: "date | None" = None) -> tuple:
    """
    Собирает историю по всем источникам за указанный период.

    Без аргументов берёт текущий месяц по сегодняшний день. Явный
    период нужен, чтобы досчитать прошлый месяц целиком: отчёты
    закрываются с задержкой, и последние числа месяца появляются
    в таблицах уже в следующем.
    """
    if d_to is None:
        d_to = REPORT_DATE or date.today()
    if d_from is None:
        d_from = d_to.replace(day=1)
    log.info("Период: %s — %s", d_from.strftime("%d.%m.%Y"),
             d_to.strftime("%d.%m.%Y"))

    fresh_h, fresh_o, reports = [], [], {}
    for src in SOURCES:
        name = src["kind"]
        src_name = {"projects": "Проекты", "alpha": "Alpha",
                    "omega": "Omega"}[name]
        known = cache_hourly.cached_dates(src_name)

        # День пропускаем, только если в кэше есть И часы, И операторы.
        # Иначе достаточно один раз потерять кэш операторов — и они
        # больше не появятся: часы-то на месте, значит читать нечего.
        # Для пяти проектов операторов пишет monitoring.ipynb, поэтому их
        # отсутствие здесь не повод перечитывать листы заново.
        if name in ("alpha", "omega"):
            known = known & cache_ops.cached_dates(src_name)

        h, o, rep = fetch_source(src, d_from, d_to, known)
        reports[name] = rep
        log.info("%s: %s", name, rep)
        if not h.empty:
            fresh_h.append(h)
        if not o.empty:
            fresh_o.append(o)

    hourly = cache_hourly.merge(
        pd.concat(fresh_h, ignore_index=True) if fresh_h else pd.DataFrame())
    ops = cache_ops.merge(
        pd.concat(fresh_o, ignore_index=True) if fresh_o else pd.DataFrame())

    if not hourly.empty:
        hourly = hourly[(hourly["Дата"] >= d_from) & (hourly["Дата"] <= d_to)]
    if not ops.empty:
        ops = ops[(ops["Дата"] >= d_from) & (ops["Дата"] <= d_to)]

    return hourly, ops, reports, d_from, d_to


hourly, operators, reports, D_FROM, D_TO = collect_history()

# Уровень обслуживания: только из кэша, из Sheets его не достать
service_level = cache_sl.load()
if not service_level.empty:
    service_level = service_level[(service_level["Дата"] >= D_FROM)
                                  & (service_level["Дата"] <= D_TO)]

print(f"\nПериод: {D_FROM.strftime('%d.%m.%Y')} — {D_TO.strftime('%d.%m.%Y')}")
print(f"Почасовых записей: {len(hourly)}, операторских: {len(operators)}, "
      f"замеров уровня обслуживания: {len(service_level)}")
if not operators.empty:
    print("\nОператоры есть по проектам:")
    for src, g in operators.groupby("Источник"):
        print(f"   {src:<20} {g['Оператор'].nunique():>3} человек, "
              f"{g['Дата'].nunique():>2} дней")
if not hourly.empty:
    print("\nДней по источникам:")
    print(hourly.groupby("Источник")["Дата"].nunique().to_string())

## ЯЧЕЙКА 3 — РАСЧЁТЫ

In [ ]:
WEEKDAYS = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]

# Порядок проектов в отчётах: сначала пятёрка, затем банки.
# Каждый проект считается отдельно — складывать Gamma с Delta
# бессмысленно, у них разница в объёме на порядки.
ENTITY_ORDER = PROJECT_ORDER + ["Alpha", "Omega"]

# Ниже этого объёма проценты — статистический шум, а не показатель.
# У проекта с двумя звонками в день «100% пропусков» означает один
# пропущенный звонок, и делать из этого вывод нельзя.
MIN_CALLS_FOR_SHARE = 30      # звонков в окне, чтобы считать долю
MIN_CALLS_FOR_TREND = 100     # звонков за период, чтобы верить тренду
MIN_LOAD_FOR_PEAK = 1.0       # входящих в час, чтобы называть его пиком


def _order_key(name: str) -> int:
    """Позиция проекта в отчёте, неизвестные уходят в конец."""
    return ENTITY_ORDER.index(name) if name in ENTITY_ORDER else 99


def _daily(hourly: pd.DataFrame) -> pd.DataFrame:
    """
    Дневная сводка ПО КАЖДОМУ ПРОЕКТУ отдельно, с показателями качества.

    Раньше пять проектов складывались в одну строку «Проекты», и итог
    оказывался почти целиком крупнейшим проектом — он даёт большую часть потока.

    Показатели названы так, как принято в контакт-центрах:

    Обращений — всё, что пришло на линию: принятые плюс пропущенные.
    Уровень ответа — доля принятых от всех входящих. Главный показатель
        качества: сколько людей дозвонились с первого раза.
    Покрытие — доля пропущенных, по которым перезвонили. Оценка
        сверху: часть исходящих может быть обычным обзвоном.
    Закрыто — доля звонков, где человек в итоге получил ответ:
        принят сразу или закрыт обратным звонком. Показывает не
        качество приёма, а результат для клиента.
    """
    if hourly.empty:
        return pd.DataFrame()

    g = hourly.groupby(["Дата", "Проект"], as_index=False).agg(
        Источник=("Источник", "first"),
        Принято=("Входящие", "sum"),
        Исходящие=("Исходящие", "sum"),
        Пропущено=("Потерянные", "sum"),
        ЕстьПропущенные=("Потерянные", lambda x: x.notna().any()),
    )
    g.loc[~g["ЕстьПропущенные"], "Пропущено"] = np.nan
    g["Обращений"] = g["Принято"] + g["Пропущено"].fillna(0)

    ok = g["ЕстьПропущенные"] & (g["Обращений"] > 0)
    g["Уровень ответа"] = np.where(
        ok, g["Принято"] / g["Обращений"].replace(0, np.nan), np.nan)
    g["Доля пропущенных"] = np.where(
        ok, g["Пропущено"] / g["Обращений"].replace(0, np.nan), np.nan)
    # Перезвонили. Делить исходящие на пропущенные напрямую нельзя:
    # исходящие включают обзвон по своей инициативе, и доля вылезает
    # за сто процентов (на реальных днях доходило до сотен процентов). Поэтому
    # в процентах показываем то, что вообще может быть закрыто
    # перезвоном — не больше, чем было пропущено.
    covered = np.minimum(g["Исходящие"], g["Пропущено"].fillna(0))
    g["Перезвонили"] = np.where(
        g["ЕстьПропущенные"] & (g["Пропущено"] > 0),
        covered / g["Пропущено"].replace(0, np.nan), np.nan)

    # А сам масштаб обзвона показываем отношением, а не долей: выше
    # единицы означает, что звонили не только по пропущенным.
    g["Исходящих на пропущенный"] = np.where(
        g["ЕстьПропущенные"] & (g["Пропущено"] > 0),
        g["Исходящие"] / g["Пропущено"].replace(0, np.nan), np.nan)

    g["Не перезвонили"] = np.where(
        g["ЕстьПропущенные"],
        (g["Пропущено"] - g["Исходящие"]).clip(lower=0), np.nan)

    g["День недели"] = g["Дата"].apply(lambda d: WEEKDAYS[d.weekday()])
    g["Тип дня"] = g["Дата"].apply(
        lambda d: "выходной" if d.weekday() >= 5 else "будний")
    g["_ord"] = g["Проект"].apply(_order_key)
    return g.sort_values(["_ord", "Проект", "Дата"]).reset_index(drop=True)


def attach_service_level(daily: pd.DataFrame, sl: pd.DataFrame,
                         hourly: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет уровень обслуживания из АТС.

    АТС считает его по часам. Среднее по часам брать нельзя: тихая
    ночь со стопроцентным уровнем весит столько же, сколько загруженный
    полдень. Поэтому берём среднее, взвешенное по звонкам того часа.
    """
    if daily.empty or sl is None or sl.empty:
        daily["SL АТС"] = np.nan
        return daily

    w = sl.merge(
        hourly.groupby(["Дата", "Проект", "Час"], as_index=False)
        .agg(вес=("Входящие", "sum")),
        left_on=["Дата", "Источник", "Час"],
        right_on=["Дата", "Проект", "Час"], how="left")
    w["вес"] = w["вес"].fillna(0)
    w = w[w["вес"] > 0]
    if w.empty:
        daily["SL АТС"] = np.nan
        return daily

    w["_num"] = w["SL"] * w["вес"]
    agg = w.groupby(["Дата", "Источник"], as_index=False).agg(
        _n=("_num", "sum"), _d=("вес", "sum"))
    agg["SL АТС"] = (agg["_n"] / agg["_d"].replace(0, np.nan)) / 100
    agg = agg.rename(columns={"Источник": "Проект"})[
        ["Дата", "Проект", "SL АТС"]]
    return daily.merge(agg, on=["Дата", "Проект"], how="left")


def _share(df: pd.DataFrame, num: str, den: str) -> float:
    """
    Доля считается как отношение сумм, а не как среднее дневных долей.

    Разница принципиальная: при среднем дневных долей тихий выходной
    с десятком звонков весит столько же, сколько загруженный вторник
    с сотней, и общая цифра уезжает на несколько пунктов вверх.
    """
    d = df[den].sum()
    return float(df[num].sum() / d) if d else np.nan


def _trend(series: pd.Series) -> float:
    """
    Наклон линейного тренда: на сколько показатель меняется за день.

    Считается методом наименьших квадратов. Плюс — растёт, минус — падает.
    Меньше трёх точек — тренда нет.
    """
    y = series.dropna()
    if len(y) < 3:
        return np.nan
    x = np.arange(len(y), dtype=float)
    return float(np.polyfit(x, y.values.astype(float), 1)[0])


def _windows(g: pd.DataFrame, window: int = 7) -> tuple:
    """
    Делит историю проекта на два последних окна равной длины.

    Окна берутся подряд по календарю, поэтому в каждое попадает
    одинаковое число выходных — сравнение остаётся честным.
    """
    g = g.sort_values("Дата")
    last = g.tail(window)
    prev = g.iloc[max(0, len(g) - 2 * window): len(g) - window]
    return last, prev


def build_summary(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Итоги по каждому проекту: где выросли, где просели.

    Будни и выходные показаны отдельными колонками — у контакт-центра
    это два разных режима, и общее среднее не описывает ни один из них.
    Тренд считается только по будним дням: иначе регулярные провалы
    выходных выдаются за колебания нагрузки.
    """
    if daily.empty:
        return pd.DataFrame()

    rows = []
    daily = daily.copy()
    # Вспомогательная колонка: сколько пропущенных реально могло быть
    # закрыто перезвоном. Нужна, чтобы доля не вылезала за сто процентов.
    daily["_Перезвонили"] = np.minimum(daily["Исходящие"],
                                       daily["Пропущено"].fillna(0))

    for proj in sorted(daily["Проект"].unique(), key=_order_key):
        g = daily[daily["Проект"] == proj]
        work = g[g["Тип дня"] == "будний"]
        rest = g[g["Тип дня"] == "выходной"]
        last, prev = _windows(g)
        has_miss = bool(g["ЕстьПропущенные"].any())

        def line(name, col, unit, growth_bad, num=None, den=None):
            """growth_bad=None — показатель нейтральный, оценки не будет."""
            if unit == "%":
                total = _share(g, num, den) * 100
                w_val = _share(work, num, den) * 100 if len(work) else np.nan
                r_val = _share(rest, num, den) * 100 if len(rest) else np.nan
                cur = _share(last, num, den) * 100 if len(last) else np.nan
                was = _share(prev, num, den) * 100 if len(prev) else np.nan
                small = 0.5
            else:
                total = g[col].sum()
                w_val = work[col].mean() if len(work) else np.nan
                r_val = rest[col].mean() if len(rest) else np.nan
                cur = last[col].mean() if len(last) else np.nan
                was = prev[col].mean() if len(prev) else np.nan
                small = max(1.0, abs(was) * 0.02) if pd.notna(was) else 1.0

            delta = cur - was if pd.notna(cur) and pd.notna(was) else np.nan

            # Для долей проверяем, на скольких звонках они посчитаны:
            # процент от пяти звонков ничего не значит
            thin = False
            if unit == "%":
                thin = (last[den].sum() < MIN_CALLS_FOR_SHARE
                        or prev[den].sum() < MIN_CALLS_FOR_SHARE)

            if pd.isna(delta):
                verdict = "мало данных"
            elif thin:
                verdict = "мало данных для %"
            elif abs(delta) < small:
                verdict = "≈ так же"
            elif growth_bad is None:
                verdict = "▲ рост" if delta > 0 else "▼ снижение"
            elif growth_bad:
                verdict = "▲ хуже" if delta > 0 else "▼ лучше"
            else:
                verdict = "▲ лучше" if delta > 0 else "▼ хуже"

            # Тренд только по будням. Для долей берётся готовая дневная
            # колонка: складывать проценты нельзя, а наклон по ним осмыслен
            trend_col = col if col else {"Пропущено": "Доля пропущенных",
                                         "Исходящие": "Перезвонили"}.get(num)
            enough = g["Обращений"].sum() >= MIN_CALLS_FOR_TREND
            slope = (_trend(work[trend_col])
                     if trend_col and len(work) >= 3 and enough else np.nan)
            k = 100.0 if unit == "%" else 1.0

            def r(v, nd=1):
                return round(v, nd) if pd.notna(v) else "—"

            return {
                "Проект": proj,
                "Показатель": name,
                "Ед.": unit,
                "Всего за период": r(total, 0 if unit != "%" else 1),
                "Будний день, среднее": r(w_val),
                "Выходной, среднее": r(r_val),
                "Последние 7 дней": r(cur),
                "Предыдущие 7 дней": r(was),
                "Изменение": r(delta),
                "Куда движется": verdict,
                "Тренд по будням, в день": r(slope * k, 2)
                                           if pd.notna(slope) else "—",
            }

        rows.append(line("Обращений всего", "Обращений", "шт", None))
        rows.append(line("Принято звонков", "Принято", "шт", None))
        rows.append(line("Исходящие звонки", "Исходящие", "шт", None))
        if has_miss:
            rows.append(line("Дозвонились с 1 раза", None, "%", False,
                             num="Принято", den="Обращений"))
            rows.append(line("Пропущено", "Пропущено", "шт", True))
            rows.append(line("Не ответили, доля от всех входящих", None, "%", True,
                             num="Пропущено", den="Обращений"))
            # Доля считается от ограниченного значения: перезвонить
            # можно не больше, чем было пропущено
            rows.append(line("Перезвонили из пропущенных", None, "%", False,
                             num="_Перезвонили", den="Пропущено"))
            rows.append(line("Не перезвонили", "Не перезвонили", "шт", True))

    return pd.DataFrame(rows)


def build_dynamics(daily: pd.DataFrame) -> pd.DataFrame:
    """
    День за днём по каждому проекту: рост, падение и необычные дни.

    Необычность считается отдельно для будней и отдельно для выходных.
    Смысл в том, что у выходного своя норма: если мерить его по общей
    медиане, каждое воскресенье будет выглядеть провалом и заслонит
    настоящие сбои. И наоборот — просевший вторник на фоне заниженной
    общей медианы может не выделиться вовсе.

    Мера отклонения — медиана и MAD, а не среднее со стандартным
    отклонением: одиночный выброс не должен раздувать порог и прятать
    соседние аномалии.
    """
    if daily.empty:
        return pd.DataFrame()

    out = []
    for proj in sorted(daily["Проект"].unique(), key=_order_key):
        g = daily[daily["Проект"] == proj].sort_values("Дата").copy()

        g["Принято: ± ко вчера"] = g["Принято"].diff()
        g["Принято: среднее 7 дней"] = (g["Принято"]
                                         .rolling(7, min_periods=2)
                                         .mean().round(1))
        g["Доля пропущенных"] = g["Доля пропущенных"].round(4)
        g["Уровень ответа"] = g["Уровень ответа"].round(4)
        g["Дозвон, изменение"] = (g["Уровень ответа"].diff() * 100).round(1)
        g["Исходящих на пропущенный"] = g["Исходящих на пропущенный"].round(2)
        if "SL АТС" in g.columns:
            g["SL АТС"] = g["SL АТС"].round(4)
        else:
            g["SL АТС"] = np.nan
        g["Обращений"] = g["Обращений"].fillna(0).astype(int)
        g["Перезвонили"] = g["Перезвонили"].round(4)
        g["Перезвон, изменение"] = (g["Перезвонили"].diff() * 100).round(1)

        # Своя норма для будней и для выходных
        g["Необычность"] = np.nan
        for kind, part in g.groupby("Тип дня"):
            if len(part) < 3:
                continue
            med = part["Принято"].median()
            mad = (part["Принято"] - med).abs().median()
            scale = mad * 1.4826 if mad > 0 else part["Принято"].std()
            if not scale or scale <= 0:
                continue
            g.loc[part.index, "Необычность"] = (
                (part["Принято"] - med) / scale).round(1)

        def describe(r):
            z = r["Необычность"]
            if pd.isna(z):
                return ""
            if z >= 2.5:
                return "звонков больше обычного"
            if z <= -2.5:
                return "звонков меньше обычного"
            if pd.notna(r["Перезвонили"]) and r["Перезвонили"] < 0.2 \
                    and r["Пропущено"] > 10:
                return "почти не перезванивали"
            return ""

        g["Что происходит"] = g.apply(describe, axis=1)
        g["Дата"] = g["Дата"].apply(lambda d: d.strftime("%d.%m.%Y"))

        out.append(g[["Проект", "Дата", "День недели", "Тип дня",
                      "Обращений", "Принято", "Принято: ± ко вчера",
                      "Принято: среднее 7 дней",
                      "Пропущено", "Уровень ответа", "Дозвон, изменение",
                      "Доля пропущенных", "Исходящие", "Перезвонили",
                      "Перезвон, изменение", "Исходящих на пропущенный",
                      "Не перезвонили", "SL АТС",
                      "Необычность", "Что происходит"]])

    res = pd.concat(out, ignore_index=True)
    res["Принято"] = res["Принято"].fillna(0).astype(int)
    res["Исходящие"] = res["Исходящие"].fillna(0).astype(int)
    for c in ["Пропущено", "Не перезвонили"]:
        res[c] = res[c].apply(lambda v: "" if pd.isna(v) else int(v))
    return res


def build_load(hourly: pd.DataFrame, daily: pd.DataFrame) -> pd.DataFrame:
    """
    Профиль суток по каждому проекту отдельно, только по будним дням.

    Выходные исключены намеренно: в них другой режим работы, и если
    смешать, средний час размажется между двумя разными картинами и
    не покажет ни одну.

    Колонка «Оценка часа» сравнивает час с собственным средним этого же
    проекта, а не с общим по компании. Иначе у мелкого проекта любой
    час выглядел бы благополучным на фоне Gamma.
    """
    if hourly.empty:
        return pd.DataFrame()

    h = hourly.copy()
    h["Будний"] = h["Дата"].apply(lambda d: d.weekday() < 5)
    h = h[h["Будний"]]
    if h.empty:
        return pd.DataFrame()

    rows = []
    for proj in sorted(h["Проект"].unique(), key=_order_key):
        p = h[h["Проект"] == proj]
        n_days = p["Дата"].nunique()
        if not n_days:
            continue

        agg = p.groupby("Час").agg(
            Входящие=("Входящие", "sum"),
            Пропущенные=("Потерянные", "sum"),
            Исходящие=("Исходящие", "sum"),
            ЕстьПропущенные=("Потерянные", lambda x: x.notna().any()),
        ).reindex(HOURS, fill_value=0)

        per_day = (agg["Входящие"] / n_days).round(1)
        share_day = (agg["Входящие"] / agg["Входящие"].sum()).round(4) \
            if agg["Входящие"].sum() else pd.Series(0.0, index=HOURS)

        has_miss = bool(agg["ЕстьПропущенные"].any())
        total_calls = agg["Входящие"] + agg["Пропущенные"]
        miss_share = (agg["Пропущенные"] / total_calls.replace(0, np.nan)) \
            if has_miss else pd.Series(np.nan, index=HOURS)
        base = float(agg["Пропущенные"].sum() / total_calls.sum()) \
            if has_miss and total_calls.sum() else np.nan

        for hh in HOURS:
            v = miss_share.get(hh, np.nan)
            load = per_day.get(hh, 0)
            if not has_miss or pd.isna(v) or load < 0.5:
                verdict = "звонков почти нет" if load < 0.5 else "нет данных"
            elif v >= base * 1.25:
                verdict = "не отвечаем чаще обычного"
            elif v <= base * 0.75:
                verdict = "справляемся лучше обычного"
            else:
                verdict = "как в среднем по проекту"

            rows.append({
                "Проект": proj,
                "Час": TIME_LABELS[hh],
                "Входящих за будний день, среднее": load,
                "Доля часа в сутках": share_day.get(hh, 0),
                "Пропущено в этот час, %": round(v, 4) if pd.notna(v) else "",
                "Оценка часа": verdict,
            })

    return pd.DataFrame(rows)


def build_operators(ops: pd.DataFrame) -> pd.DataFrame:
    """
    Кто сколько звонков берёт внутри своего проекта.

    Оценка идёт по МЕСТУ в команде, а не по проценту от медианы.
    Причина: у медианы два изъяна. Она даёт числа вроде «+272%»,
    которые глазами не читаются, и она не различает людей —
    достаточно, чтобы кто-то один работал хуже тебя, и ты уже
    «как у остальных», даже если сделал мало.

    Место говорит прямо: третий из двенадцати. А словесная оценка
    делит команду на три части — верхняя треть, середина, нижняя.
    В командах меньше четырёх человек оценка не ставится: сравнивать
    не с чем.
    """
    if ops.empty:
        return pd.DataFrame()

    per_day = ops.groupby(["Источник", "Оператор", "Дата"], as_index=False).agg(
        Принято=("Принято", "sum"), Секунды=("Секунды", "sum"),
        Звонков=("Звонков", "sum"))
    per_day = per_day[per_day["Принято"] > 0]
    if per_day.empty:
        return pd.DataFrame()

    g = per_day.groupby(["Источник", "Оператор"], as_index=False).agg(
        Дней=("Дата", "nunique"), Принято=("Принято", "sum"),
        Секунды=("Секунды", "sum"), Звонков=("Звонков", "sum"),
        Разброс=("Принято", "std"))

    g["Звонков за смену"] = (g["Принято"] / g["Дней"]).round(1)
    g["Средний разговор, сек"] = (
        g["Секунды"] / g["Звонков"].replace(0, np.nan)).round(0)

    total = g.groupby("Источник")["Принято"].transform("sum")
    g["Доля в общем объёме"] = (g["Принято"] / total).round(3)

    # Место в команде: первое у того, кто берёт больше всех звонков
    # за смену. К номеру оператора отношения не имеет — Operator12
    # вполне может быть первым, а Operator1 последним.
    g["Место"] = (g.groupby("Источник")["Звонков за смену"]
                   .rank(method="min", ascending=False).astype(int))
    g["Человек в команде"] = g.groupby("Источник")["Оператор"].transform("size")

    # Во сколько раз от середины команды. Множитель вместо процентов:
    # «2,4 раза» читается сразу, «+140%» — нет.
    mid = g.groupby("Источник")["Звонков за смену"].transform("median")
    g["Во сколько раз от середины"] = (
        g["Звонков за смену"] / mid.replace(0, np.nan)).round(2)

    def verdict(row):
        """
        Словесная оценка по месту в команде, а не по проценту.

        Делим на три части: верхняя треть берёт много, нижняя мало,
        остальные посередине. При команде меньше четырёх человек
        любое деление бессмысленно — так и пишем.
        """
        n = row["Человек в команде"]
        if n < 4:
            return "мало людей для сравнения"
        place = row["Место"]
        if place <= n / 3:
            return "берёт много"
        if place > n * 2 / 3:
            return "берёт мало"
        return "как большинство"

    g["Сколько берёт"] = g.apply(verdict, axis=1)

    # Проверка качества: разговор заметно короче, чем у команды.
    # Иначе первое место можно занять, обрывая звонки — взял много,
    # поговорил ни о чём. Считается от медианы команды, потому что
    # у разных проектов своя нормальная длина разговора.
    mid_talk = g.groupby("Источник")["Средний разговор, сек"].transform("median")
    short = g["Средний разговор, сек"] < mid_talk * 0.6
    g["Качество разговора"] = np.where(
        g["Средний разговор, сек"].isna(), "",
        np.where(short, "разговоры коротковаты", "разговоры обычной длины"))

    # Ровно ли работает: разброс относительно своей же нормы.
    # У одного каждый день по сорок звонков, у другого то восемьдесят,
    # то пять — среднее одинаковое, поведение разное.
    g["Ровность работы"] = np.where(
        g["Дней"] < 3, "",
        np.where(g["Разброс"] / g["Звонков за смену"].replace(0, np.nan) > 0.5,
                 "работает неровно", "работает ровно"))

    g = g.rename(columns={"Источник": "Проект", "Принято": "Принято всего"})
    return g[["Проект", "Место", "Оператор", "Человек в команде",
              "Принято всего", "Звонков за смену", "Дней",
              "Средний разговор, сек", "Доля в общем объёме",
              "Во сколько раз от середины", "Сколько берёт",
              "Качество разговора", "Ровность работы"]].sort_values(
        ["Проект", "Место"], key=lambda c: (c.map(_order_key)
                                            if c.name == "Проект" else c)
    ).reset_index(drop=True)


def build_anomalies(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Ищет дни, которые заметно выбиваются из обычной картины.

    Аномалия — это редкое событие. Если в отчёте их десяток за две
    недели, значит порог занижен и в список попал обычный разброс.
    Поэтому здесь три фильтра, и день должен пройти все:

      1. Сильно отклонился по разбросу самого проекта (не меньше
         трёх сигм). Медиана и MAD вместо среднего и стандартного
         отклонения — их не сносит один выброс.
      2. Отклонился заметно и в абсолютных числах. Иначе у мелкого
         проекта прыжок с 2 до 8 звонков выглядит катастрофой.
      3. Норма посчитана хотя бы по восьми похожим дням. На четырёх
         точках разброс считать нельзя, там что угодно аномалия.

    Будни и выходные сравниваются отдельно: у выходного своя картина,
    и по общей норме каждое воскресенье выглядело бы провалом.

    Подряд идущие дни склеиваются в одно событие: три дня падения —
    это одна проблема, а не три аномалии.
    """
    if daily.empty:
        return pd.DataFrame()

    SIGMA = 3.0            # насколько сильно должно отклониться
    MIN_REL = 0.30         # и хотя бы на треть от нормы
    MIN_DAYS = 8           # столько похожих дней нужно для нормы
    TOP_PER_PROJECT = 3    # больше и не покажем: это уже не редкость

    found = []
    for proj in sorted(daily["Проект"].unique(), key=_order_key):
        g = daily[daily["Проект"] == proj].sort_values("Дата")
        if g["Обращений"].sum() < MIN_CALLS_FOR_TREND:
            continue

        hits = []
        for kind, part in g.groupby("Тип дня"):
            if len(part) < MIN_DAYS:
                continue

            def norm(col):
                """Норма и разброс по медиане и MAD."""
                v = part[col].dropna()
                if len(v) < MIN_DAYS:
                    return None, None
                med = v.median()
                mad = (v - med).abs().median()
                sc = mad * 1.4826 if mad > 0 else v.std()
                return (med, sc) if sc and sc > 0 else (med, None)

            checks = [
                ("Уровень ответа", "дозвонились намного реже обычного",
                 "down", "люди не дозвонились до нас"),
                ("Обращений", "звонков пришло намного больше обычного",
                 "up", "поток вырос — надо смотреть, хватило ли людей"),
                ("Обращений", "звонков пришло намного меньше обычного",
                 "down", "поток упал — возможно, что-то с линией"),
                ("Перезвонили", "перезванивали намного реже обычного",
                 "down", "по пропущенным перестали перезванивать"),
            ]
            for col, label, side, why in checks:
                if col not in part.columns:
                    continue
                med, sc = norm(col)
                if sc is None or not med:
                    continue

                for _, r in part.iterrows():
                    val = r[col]
                    if pd.isna(val):
                        continue
                    z = (val - med) / sc
                    rel = abs(val - med) / abs(med) if med else 0
                    # Оба условия сразу: и по разбросу, и по величине.
                    if rel < MIN_REL:
                        continue
                    if side == "up" and z < SIGMA:
                        continue
                    if side == "down" and z > -SIGMA:
                        continue

                    share = col in ("Уровень ответа", "Перезвонили")
                    hits.append({
                        "Проект": proj, "_d": r["Дата"], "_col": col,
                        "Что случилось": label,
                        "В этот день": (f"{val * 100:.0f}%" if share
                                        else f"{val:.0f}"),
                        "Обычно": (f"{med * 100:.0f}%" if share
                                   else f"{med:.0f}"),
                        "Насколько сильно": ("очень сильно" if abs(z) >= 5
                                             else "заметно"),
                        "Что это значит": why,
                        "_z": abs(z),
                    })

        # Склеиваем подряд идущие дни по одному и тому же поводу.
        by_kind = {}
        for h in hits:
            by_kind.setdefault((h["_col"], h["Что случилось"]), []).append(h)

        events = []
        for key, items in by_kind.items():
            items.sort(key=lambda x: x["_d"])
            run = [items[0]]
            for cur in items[1:]:
                if (cur["_d"] - run[-1]["_d"]).days <= 1:
                    run.append(cur)
                else:
                    events.append(run)
                    run = [cur]
            events.append(run)

        for run in events:
            strongest = max(run, key=lambda x: x["_z"])
            first, last = run[0]["_d"], run[-1]["_d"]
            when = (first.strftime("%d.%m.%Y") if first == last
                    else f"{first:%d.%m} — {last:%d.%m.%Y}")
            row = dict(strongest)
            row["Когда"] = when
            row["Дней подряд"] = len(run)
            if len(run) > 1:
                row["Насколько сильно"] = f"{len(run)} дня подряд"
            found.append(row)

    if not found:
        return pd.DataFrame()

    df = pd.DataFrame(found)
    # На проект оставляем только самые сильные события: аномалия по
    # определению редкость, длинный список ей противоречит.
    df = (df.sort_values("_z", ascending=False)
            .groupby("Проект", group_keys=False)
            .head(TOP_PER_PROJECT))
    df["Дата"] = df["_d"]
    df = df.sort_values(["Проект", "_d"])
    return df[["Проект", "Когда", "Что случилось", "В этот день",
               "Обычно", "Насколько сильно", "Что это значит",
               "Дата"]].reset_index(drop=True)


def build_notes(daily: pd.DataFrame, load: pd.DataFrame,
                ops: pd.DataFrame, d_from: date, d_to: date) -> pd.DataFrame:
    """
    Выводы обычным языком, по одному блоку на проект.

    Каждое наблюдение опирается на конкретные числа из соседних вкладок,
    чтобы его можно было перепроверить, а не принимать на веру.
    """
    notes = []

    def add(proj, tema, text):
        notes.append({"Проект": proj, "Тема": tema, "Что видно": text})

    add("— Период —", "Охват",
        f"Данные с {d_from.strftime('%d.%m.%Y')} по {d_to.strftime('%d.%m.%Y')}. "
        f"Дней в выборке: {daily['Дата'].nunique()}. "
        f"Проектов: {daily['Проект'].nunique()}.")

    for proj in sorted(daily["Проект"].unique(), key=_order_key):
        g = daily[daily["Проект"] == proj]
        work = g[g["Тип дня"] == "будний"]
        last, prev = _windows(g)
        has_miss = bool(g["ЕстьПропущенные"].any())

        # Мелкие проекты описываем одной строкой и не делаем из их
        # процентов выводов: там любое движение — один-два звонка
        volume = g["Обращений"].sum()
        if volume < MIN_CALLS_FOR_TREND:
            add(proj, "Объём",
                f"За период всего {int(volume)} входящих звонков "
                f"({work['Принято'].mean():.1f} входящих в будний день). "
                f"Слишком мало, чтобы говорить о процентах и трендах — "
                f"смотрите абсолютные числа во вкладке «Итоги».")
            continue

        if len(work) >= 2:
            add(proj, "Масштаб",
                f"В будний день в среднем {work['Принято'].mean():.0f} входящих, "
                f"в выходной {g[g['Тип дня']=='выходной']['Принято'].mean():.0f}."
                if (g["Тип дня"] == "выходной").any() else
                f"В будний день в среднем {work['Принято'].mean():.0f} входящих.")

        if len(last) and len(prev):
            cur, was = last["Принято"].mean(), prev["Принято"].mean()
            if was and abs((cur - was) / was) >= 0.08:
                ch = (cur - was) / was
                add(proj, "Поток",
                    f"Входящих {'больше' if ch > 0 else 'меньше'} на "
                    f"{abs(ch)*100:.0f}%: {was:.0f} → {cur:.0f} в день "
                    f"(последняя неделя против предыдущей).")

        if has_miss:
            ar = _share(g, "Принято", "Обращений")
            if pd.notna(ar):
                verdict = ("хорошо" if ar >= 0.9 else
                           "приемлемо" if ar >= 0.8 else
                           "низко" if ar >= 0.6 else "плохо")
                add(proj, "Уровень ответа",
                    f"С первого раза дозвонились {ar*100:.0f}% обратившихся "
                    f"— {verdict}. Остальные попали в пропущенные.")
                l1 = _share(last, "Принято", "Обращений")
                l0 = _share(prev, "Принято", "Обращений")
                if pd.notna(l1) and pd.notna(l0) and abs((l1 - l0) * 100) >= 2:
                    dp = (l1 - l0) * 100
                    add(proj, "Уровень ответа",
                        f"За неделю {'вырос' if dp > 0 else 'упал'} на "
                        f"с {l0*100:.0f}% до {l1*100:.0f}%.")

            if "SL АТС" in g.columns and g["SL АТС"].notna().any():
                sl = g["SL АТС"].mean()
                add(proj, "Уровень обслуживания",
                    f"По данным АТС — {sl*100:.0f}%. Это его собственный "
                    f"расчёт, взвешенный по загрузке часов.")

            miss, back = g["Пропущено"].sum(), g["Исходящие"].sum()
            if miss:
                cov = min(back, miss) / miss
                tail = int(max(0, miss - back))
                if cov >= 0.8:
                    verdict = "почти по всем перезванивают"
                elif cov >= 0.5:
                    verdict = "перезванивают примерно по половине"
                else:
                    verdict = "большинству обратно не звонят"
                add(proj, "Обратные звонки",
                    f"Пропущено {int(miss)}, исходящих {int(back)} — "
                    f"покрытие {cov*100:.0f}%, {verdict}. "
                    f"Без обратного звонка около {tail} звонков.")

            if len(last) and len(prev):
                c1 = _share(last, "Исходящие", "Пропущено")
                c0 = _share(prev, "Исходящие", "Пропущено")
                if pd.notna(c1) and pd.notna(c0) and abs((c1 - c0) * 100) >= 5:
                    d_pp = (c1 - c0) * 100
                    add(proj, "Обратные звонки",
                        f"Покрытие {'выросло' if d_pp > 0 else 'упало'} на "
                        f"за неделю: "
                        f"{c0*100:.0f}% → {c1*100:.0f}%.")

                s1 = _share(last, "Пропущено", "Обращений")
                s0 = _share(prev, "Пропущено", "Обращений")
                if pd.notna(s1) and pd.notna(s0) and abs((s1 - s0) * 100) >= 1:
                    d_pp = (s1 - s0) * 100
                    add(proj, "Пропуски",
                        f"Доля пропущенных "
                        f"{'выросла' if d_pp > 0 else 'снизилась'} на "
                        f"с {s0*100:.1f}% до {s1*100:.1f}%.")

            wm = work.dropna(subset=["Доля пропущенных"])
            wm = wm[wm["Обращений"] >= MIN_CALLS_FOR_SHARE]
            if len(wm) >= 3:
                worst = wm.loc[wm["Доля пропущенных"].idxmax()]
                best = wm.loc[wm["Доля пропущенных"].idxmin()]
                add(proj, "Пропуски",
                    f"Из будних дней тяжелее всего "
                    f"{worst['Дата'].strftime('%d.%m')} "
                    f"({worst['Доля пропущенных']*100:.0f}% пропусков), "
                    f"легче всего {best['Дата'].strftime('%d.%m')} "
                    f"({best['Доля пропущенных']*100:.0f}%).")

            slope = _trend(work["Доля пропущенных"]) if len(work) >= 3 else np.nan
            if pd.notna(slope) and abs(slope) * 100 >= 0.2:
                add(proj, "Тренд",
                    f"По будням доля пропущенных "
                    f"{'растёт' if slope > 0 else 'снижается'} примерно на "
                    f"примерно на {abs(slope)*100:.1f}% в день.")

        if not load.empty:
            lp = load[load["Проект"] == proj]
            bad = lp[lp["Оценка часа"] == "не отвечаем чаще обычного"]
            if not bad.empty:
                add(proj, "Часы",
                    "Пропусков больше обычного в часы: "
                    + ", ".join(bad["Час"].head(6))
                    + ". Здесь не хватает людей на линии.")
            busy = lp.sort_values("Входящих за будний день, среднее",
                                  ascending=False).head(1)
            if not busy.empty and (busy.iloc[0]["Входящих за будний день, среднее"]
                                   >= MIN_LOAD_FOR_PEAK):
                r = busy.iloc[0]
                add(proj, "Часы",
                    f"Пик — {r['Час']}, "
                    f"{r['Входящих за будний день, среднее']:.0f} входящих "
                    f"в среднем за будний день.")

        if not ops.empty:
            o = ops[ops["Проект"] == proj]
            # Колонка называется «Сколько берёт», и значения в ней
            # те же, что видны на листе операторов. Раньше тут стояло
            # старое имя — лист падал с KeyError.
            for label, mark in [("берёт много", "Берут больше всех"),
                                ("берёт мало", "Берут меньше всех")]:
                sel = o[o["Сколько берёт"] == label]
                if not sel.empty:
                    add(proj, "Операторы", mark + ": " + ", ".join(
                        f"{r['Оператор']} "
                        f"({r['Звонков за смену']:.0f} за смену)"
                        for _, r in sel.iterrows()))

    return pd.DataFrame(notes)


def build_glossary() -> pd.DataFrame:
    """Расшифровка показателей — чтобы отчёт читался без пояснений автора."""
    items = [
        ("Обращений всего", "шт",
         "Принятые плюс пропущенные — весь поток, пришедший на линию."),
        ("Принято звонков", "шт",
         "Звонки, которые оператор взял и обработал."),
        ("Дозвонились с 1 раза", "%",
         "Принято ÷ все входящие. Главный показатель приёма: "
         "какая доля людей дозвонилась сразу, без повторных попыток."),
        ("Уровень обслуживания АТС", "%",
         "Собственный расчёт АТС по часам. Здесь усреднён с весом "
         "по числу звонков в каждом часе: тихая ночь со стопроцентным "
         "уровнем не должна весить как загруженный полдень."),
        ("Насколько сильно", "текст",
         "Насколько день выбился из обычной картины проекта. День "
         "попадает в «Аномалии», только если отклонился и по разбросу, "
         "и минимум на треть от нормы — иначе туда сыпался бы обычный "
         "разброс. Подряд идущие дни склеиваются в одно событие."),
        ("Пропущенные", "шт",
         "Звонки, до оператора не дошедшие. Сами по себе ещё не потеря — "
         "многие закрываются обратным звонком."),
        ("Не ответили, доля от всех входящих", "%",
         "Пропущенные ÷ (входящие + пропущенные). Считается от сумм за "
         "период, а не как среднее дневных долей."),
        ("Исходящие звонки", "шт",
         "Все исходящие, включая обратные звонки по пропущенным."),
        ("Перезвонили из пропущенных", "%",
         "Какая доля пропущенных могла быть закрыта обратным звонком. "
         "Считается от ограниченного числа: перезвонить нельзя больше, "
         "чем было пропущено, поэтому за сто процентов не выходит. "
         "Оценка сверху — какие именно исходящие были перезвонами, "
         "в данных не видно."),
        ("Исходящих на 1 пропущенный", "×",
         "Отношение, а не доля. Выше единицы означает, что звонили не "
         "только по пропущенным, но и по своей инициативе. На реальных "
         "днях доходило до 7 — поэтому это отдельная колонка, а не процент."),
        ("Во сколько раз от середины", "раз",
         "Насколько оператор выше или ниже середины своей команды. "
         "Это сравнение, а не доля, поэтому может быть и больше ста."),
        ("Не перезвонили", "шт",
         "Пропущенные минус исходящие, но не меньше нуля."),
        ("Принято: ± ко вчера", "шт",
         "Разница с предыдущим днём в выборке."),
        ("Принято: среднее 7 дней", "шт",
         "Скользящее среднее — сглаживает дневной шум и показывает уровень."),
        ("Дозвон, изменение", "пункты",
         "Изменение доли в процентных пунктах. Разница между 30% и 35% — "
         "это 5 пунктов, а не 16 процентов."),
        ("Необычность", "число",
         "На сколько день отличается от обычного. Считается по медиане и "
         "MAD отдельно для будней и выходных: у выходного своя норма. "
         "Больше 3 — звонков намного больше обычного, меньше −3 — намного меньше."),
        ("Доля часа в сутках", "%",
         "Какая часть дневного потока приходится на этот час."),
        ("Оценка часа", "текст",
         "Сравнение часа со средним по этому же проекту, а не по компании."),
        ("Принято за смену, среднее", "шт",
         "Принятые звонки ÷ число смен, а не календарных дней."),
        ("Место", "номер",
         "Насколько оператор выше или ниже середины своей команды."),
        ("Разброс по дням", "коэф.",
         "Ровность выработки. Меньше — стабильнее; большое значение "
         "обычно объясняется графиком смен."),
        ("Тренд по будням, в день", "ед./день",
         "Наклон прямой по будним дням. Выходные исключены, чтобы их "
         "регулярные провалы не выдавались за динамику."),
    ]
    return pd.DataFrame(items, columns=["Показатель", "Ед.", "Что означает"])


# ============================================================
#  ЛИСТ «ЧТО ПРОИСХОДИТ» — одна страница для супервайзера
# ============================================================

# Пороги. Вынесены сюда, чтобы менять в одном месте.
RUN_DAYS = 3            # сколько дней подряд считать устойчивым движением
LOW_ANSWER = 0.70       # уровень ответа ниже — плохо
BAD_HOUR_LOST = 0.45    # доля потерь в часе, выше которой час провальный
MIN_PER_DAY = 10        # проекты мельче не разбираем: там любое
                        # движение — один-два звонка, то есть шум
DROP_VOLUME = 0.35      # падение объёма, которое стоит проверить

P_RED, P_YELLOW, P_GREEN = "1 Срочно", "2 Внимание", "3 Норма"


def _run_length(values: list) -> int:
    """
    Сколько последних дней показатель рос подряд.

    Это и есть отличие сигнала от шума: один скачок может быть
    случайностью, три дня подряд в одну сторону — уже движение.
    """
    n = 0
    for i in range(len(values) - 1, 0, -1):
        if pd.isna(values[i]) or pd.isna(values[i - 1]):
            break
        if values[i] > values[i - 1]:
            n += 1
        else:
            break
    return n


def _days(n: int) -> str:
    """«3 дня», «5 дней» — иначе в отчёте для людей режет глаз."""
    if n % 10 == 1 and n % 100 != 11:
        return f"{n} день"
    if n % 10 in (2, 3, 4) and n % 100 not in (12, 13, 14):
        return f"{n} дня"
    return f"{n} дней"


def _spread(v: pd.Series) -> float:
    """
    Разброс по MAD — устойчивая мера, её не сносит один выброс.

    Нужна, чтобы понять, велико ли отклонение ДЛЯ ЭТОГО проекта.
    У крупного проекта 30 звонков разницы — ничто, у мелкого — вдвое.
    """
    v = v.dropna()
    if len(v) < 4:
        return np.nan
    return float((v - v.median()).abs().median()) * 1.4826


def build_brief(daily: pd.DataFrame, load: pd.DataFrame,
                anomalies: pd.DataFrame) -> pd.DataFrame:
    """
    Одна страница для супервайзера: что происходит и что делать.

    Не таблица показателей, а список наблюдений, отсортированный по
    важности. Каждое наблюдение отвечает на три вопроса: что видно,
    случайность это или нет, и что с этим делать.

    Остальные листы остаются для аналитика — сюда попадает только
    то, по чему нужно принимать решение.
    """
    if daily.empty:
        return pd.DataFrame()

    rows = []

    def add(prio, proj, what, sure, todo):
        rows.append({"Важность": prio, "Проект": proj,
                     "Что происходит": what,
                     "Случайность или нет": sure,
                     "Что сделать": todo})

    total_lost = daily["Пропущено"].sum(skipna=True)
    total_calls = daily["Обращений"].sum()
    total_out = daily["Исходящие"].sum()
    if total_calls:
        add("0 Итог", "— Все проекты —",
            f"За период поступило {int(total_calls)} входящих звонков. "
            f"Из них не ответили на {int(total_lost)} "
            f"({total_lost / total_calls:.0%}). "
            f"Исходящих сделано {int(total_out)}.",
            "Факт за весь период",
            "Ниже — по каким проектам и часам не отвечают.")

    for proj in sorted(daily["Проект"].unique()):
        g = daily[daily["Проект"] == proj].sort_values("Дата")
        calls = g["Обращений"].sum()

        if not g["ЕстьПропущенные"].any():
            add(P_GREEN, proj,
                "Нет данных о неотвеченных звонках — "
                "видно только принятые.",
                "Не с чем сравнивать",
                "Проверить, приходит ли отчёт АТС по этому проекту.")
            continue

        per_day = calls / max(len(g), 1)
        if per_day < MIN_PER_DAY:
            add(P_GREEN, proj,
                f"Мало звонков — в среднем {per_day:.0f} в день.",
                "Выводы делать рано: на таких числах любое движение "
                "укладывается в случайность",
                "Следить, но отдельных мер не нужно.")
            continue

        # --- 1. Потери растут несколько дней подряд ---
        lost = g["Доля пропущенных"].tolist()
        run = _run_length(lost)
        if run >= RUN_DAYS and not pd.isna(lost[-1]):
            was = lost[-1 - run]
            add(P_RED, proj,
                f"Всё чаще не отвечают на звонки, {_days(run)} подряд: "
                f"было {was:.0%} неотвеченных, стало {lost[-1]:.0%}.",
                f"Устойчиво — {_days(run)} в одну сторону, "
                f"это уже не колебание",
                "Посмотреть, хватает ли людей в провальные часы.")

        # --- 2. Уровень ответа устойчиво низкий ---
        week = g.tail(7)["Уровень ответа"].dropna()
        if len(week) >= 4 and week.median() < LOW_ANSWER:
            add(P_RED, proj,
                f"Отвечают только на {week.median():.0%} входящих звонков.",
                f"Устойчиво — так последние {_days(len(week))}",
                "Пересмотреть расстановку смен по этому проекту.")

        # --- 3. Резкое падение объёма ---
        last, prev = g.tail(3), g.iloc[max(0, len(g) - 10):len(g) - 3]
        if len(prev) >= 4 and prev["Обращений"].median() > 0:
            change = (last["Обращений"].median()
                      / prev["Обращений"].median() - 1)
            spread = _spread(prev["Обращений"])
            moved = abs(last["Обращений"].median()
                        - prev["Обращений"].median())
            # Падение засчитываем, только если оно больше обычного
            # разброса: иначе поймаем обычные качели спроса.
            if change < -DROP_VOLUME and (pd.isna(spread) or moved > spread):
                add(P_YELLOW, proj,
                    f"Входящих звонков стало меньше на {abs(change):.0%}: "
                    f"было {prev['Обращений'].median():.0f} в день, "
                    f"стало {last['Обращений'].median():.0f}.",
                    "Больше обычных колебаний",
                    "Проверить, работает ли линия и не изменился ли поток.")

        # --- 4. Свежие аномалии ---
        if not anomalies.empty and "Проект" in anomalies:
            recent = anomalies[anomalies["Проект"] == proj]
            if len(recent):
                last_days = set(g.tail(3)["Дата"])
                hit = recent[recent["Дата"].isin(last_days)] \
                    if "Дата" in recent else recent.iloc[0:0]
                for _, r in hit.iterrows():
                    add(P_YELLOW, proj,
                        f"{r['Дата'].strftime('%d.%m')}: "
                        f"{str(r.get('Что случилось', 'необычный день')).lower()}",
                        "Разовое отклонение, не тренд",
                        "Выяснить причину: была акция, сбой или рассылка.")

        # --- 5. Провальные часы ---
        if not load.empty and "Проект" in load:
            l = load[load["Проект"] == proj]
            # Колонку ищем по названию, а не по номеру: в таблице
            # нагрузки первым столбцом идёт проект, и слепой отсчёт
            # выводил бы название проекта вместо часа.
            hour_col = next((c for c in l.columns
                             if "час" in str(c).lower()), None)
            col = next((c for c in l.columns
                        if "пропущен" in str(c).lower()
                        and ("доля" in str(c).lower() or "%" in str(c))), None)
            if col is not None and hour_col is not None and len(l):
                bad = l[pd.to_numeric(l[col], errors="coerce") > BAD_HOUR_LOST]
                if len(bad):
                    hours = ", ".join(str(x) for x in bad[hour_col].head(4))
                    add(P_RED, proj,
                        f"Часы, когда до нас не дозваниваются: {hours}.",
                        "Повторяется изо дня в день",
                        "Сдвинуть смены на эти часы — людей не добавляя.")

        if not any(r["Проект"] == proj and r["Важность"] != P_GREEN
                   for r in rows):
            add(P_GREEN, proj, "Без заметных изменений.",
                "В пределах обычного разброса", "Мер не требуется.")

    out = pd.DataFrame(rows)
    return out.sort_values(["Важность", "Проект"]).reset_index(drop=True)


def compute_all(d_from: date, d_to: date, src=None) -> dict:
    """
    Считает все таблицы отчёта за период.

    Собрано одной функцией, чтобы тот же расчёт можно было прогнать
    и для прошлого месяца при его закрытии, а не только для текущего.

    :param src: готовые (hourly, operators, service_level); None —
                собрать заново по указанному периоду
    """
    if src is None:
        h, o, _, _, _ = collect_history(d_from, d_to)
        sl = cache_sl.load()
        if not sl.empty:
            sl = sl[(sl["Дата"] >= d_from) & (sl["Дата"] <= d_to)]
    else:
        h, o, sl = src

    d = attach_service_level(_daily(h), sl, h)
    tables = {
        "daily": d,
        "summary": build_summary(d),
        "anomalies": build_anomalies(d),
        "dynamics": build_dynamics(d),
        "load": build_load(h, d),
        "ops": build_operators(o),
        "glossary": build_glossary(),
        "период": (d_from, d_to),
    }
    tables["notes"] = build_notes(d, tables["load"], tables["ops"],
                                  d_from, d_to)
    # Собирается последним: опирается на аномалии и профиль часов.
    tables["brief"] = build_brief(d, tables["load"], tables["anomalies"])
    return tables


TABLES = compute_all(D_FROM, D_TO, src=(hourly, operators, service_level))

brief = TABLES["brief"]
daily = TABLES["daily"]
summary = TABLES["summary"]
anomalies = TABLES["anomalies"]
dynamics = TABLES["dynamics"]
load = TABLES["load"]
ops_table = TABLES["ops"]
notes = TABLES["notes"]
glossary = TABLES["glossary"]

preview(summary, f"ИТОГИ ПО ПРОЕКТАМ — {D_FROM.strftime('%d.%m')} — {D_TO.strftime('%d.%m.%Y')}")
preview(dynamics.tail(12), "ДИНАМИКА (хвост)")
preview(load.head(12), "НАГРУЗКА (начало)")
preview(ops_table, "ОПЕРАТОРЫ")
preview(anomalies.head(25), "АНОМАЛИИ")
preview(notes.head(20), "ВЫВОДЫ")

## ЯЧЕЙКА 4 — ВЫГРУЗКА

In [ ]:
# Колонки с долями — им ставится процентный формат
PERCENT_COLUMNS = {
    "Уровень ответа", "Доля пропущенных", "Перезвонили", "SL АТС",
    "Доля часа в сутках", "Пропущено в этот час, %",
    "Доля в общем объёме", "Относительно медианы команды",
}

# Рост этих колонок — плохая новость
WORSE_ON_GROWTH = {"Пропущено: ± п.п."}

# Рост этих колонок — хорошая новость
BETTER_ON_GROWTH = {"Принято: ± ко вчера", "Перезвонили: ± п.п.",
                    "Относительно медианы команды"}

# Понятные заголовки для колонок, где короткое имя не объясняет смысла
RENAME_FOR_SHEET = {
    "Уровень ответа": "Дозвонились с 1 раза, %",
    "Доля пропущенных": "Пропущено, % от обращений",
    "Перезвонили": "Перезвонили, % от пропущенных",
    "Исходящих на пропущенный": "Исходящих на 1 пропущенный, ×",
    "SL АТС": "Уровень обслуживания АТС, %",
    "Не перезвонили": "Не перезвонили, шт",
    "Пропущено": "Пропущено, шт",
    "Принято": "Принято, шт",
    "Обращений": "Обращений, шт",
    "Исходящие": "Исходящих, шт",
}


def _with_group_breaks(df: pd.DataFrame, col: str) -> tuple:
    """
    Вставляет строку-заголовок перед каждой группой.

    Так проекты не сливаются друг с другом: глазу есть за что зацепиться,
    и видно, где заканчивается один и начинается следующий.
    Возвращает таблицу и номера строк-заголовков для подсветки.
    """
    if df.empty or col not in df.columns:
        return df, []

    parts, breaks, row = [], [], 1        # строка 0 — шапка листа
    for name, g in df.groupby(col, sort=False):
        head = {c: "" for c in df.columns}
        head[col] = f"▌ {name}"
        parts.append(pd.DataFrame([head]))
        breaks.append(row)
        row += 1 + len(g)
        parts.append(g)

    return pd.concat(parts, ignore_index=True), breaks


def table_month(loader) -> "date | None":
    """
    Какой месяц сейчас лежит в таблице аналитики.

    Смотрим вкладку «По дням» — там в колонке дат видно, за какой
    период таблица собрана. Отдельного маркера не заводим: он мог бы
    разойтись с содержимым, а даты не соврут.
    """
    for w in loader.sh.worksheets():
        if "по дням" not in w.title.lower():
            continue
        try:
            vals = w.get("B1:B80")
        except Exception as e:
            log.warning("Вкладку «%s» прочитать не вышло (%s)",
                        w.title, type(e).__name__)
            return None
        for row in (vals or []):
            if not row:
                continue
            d = pd.to_datetime(str(row[0]).strip(), format="%d.%m.%Y",
                               errors="coerce")
            if pd.notna(d):
                return d.date()
    return None


def close_previous_month(loader) -> None:
    """
    Закрывает прошлый месяц: досчитывает целиком и сохраняет в Excel.

    Отчёты закрываются с задержкой — данные за последние числа месяца
    попадают в таблицы уже в следующем. Если просто выгрузить таблицу
    как есть, в архив уйдёт неполный месяц: там будет то, что успело
    посчитаться при последнем запуске, без хвоста.

    Поэтому месяц сначала пересчитывается за полный календарный срок,
    записывается в таблицу, и только потом уходит в файл.
    """
    was = table_month(loader)
    if was is None:
        log.info("В таблице пока нет данных — закрывать нечего")
        return
    if (was.year, was.month) == (D_TO.year, D_TO.month):
        return

    out = ARCHIVE_DIR / f"Аналитика {was.year} {RU_MONTHS[was.month]}.xlsx"
    if out.exists():
        log.info("Архив %s уже есть — таблицу просто перепишу", out.name)
        return

    import calendar
    p_from = was.replace(day=1)
    p_to = was.replace(day=calendar.monthrange(was.year, was.month)[1])

    print(f"\nВ таблице лежит {RU_MONTHS[was.month]} {was.year}, "
          f"а считаем {RU_MONTHS[D_TO.month]} {D_TO.year}.")
    print(f"Досчитываю {RU_MONTHS[was.month]} целиком "
          f"({p_from.strftime('%d.%m')} — {p_to.strftime('%d.%m')}) "
          f"и сохраняю в файл...")

    try:
        prev_tables = compute_all(p_from, p_to)
    except Exception as e:
        log.warning("Пересчитать %s не удалось (%s: %s). "
                    "Сохраняю таблицу как есть — проверьте полноту архива",
                    RU_MONTHS[was.month], type(e).__name__, str(e)[:120])
    else:
        d = prev_tables["daily"]
        if d.empty:
            log.warning("За %s данных не нашлось", RU_MONTHS[was.month])
        else:
            print(f"   дней собрано: {d['Дата'].nunique()}, "
                  f"последний {d['Дата'].max().strftime('%d.%m.%Y')}")
            write_tables(loader, prev_tables)

    loader.archive_to_excel(out)
    print(f"Сохранено: {out}")


def write_tables(loader, tables: dict) -> None:
    """
    Записывает готовые таблицы на вкладки.

    Вынесено отдельно, потому что тем же кодом пишется и текущий
    месяц, и прошлый при его закрытии.
    """
    # Порядок как в книге: сначала главные числа, потом их толкование,
    # затем детали и в конце словарь терминов
    blocks = [
        # Первым листом — то, что открывает супервайзер. Остальные
    # оставлены для аналитика: они нужны, чтобы перепроверить вывод,
    # а не чтобы их читали каждый день.
    ("0. Что происходит", tables["brief"], None),
    ("1. Итоги", tables["summary"], "Проект"),
        ("2. Выводы", tables["notes"], "Проект"),
        ("3. Аномалии", tables["anomalies"], "Проект"),
        ("4. По дням", tables["dynamics"], "Проект"),
        ("5. По часам", tables["load"], "Проект"),
        ("6. Операторы", tables["ops"], "Проект"),
        ("7. Справочник", tables["glossary"], None),
    ]

    # Вкладки от прежних версий отчёта удаляются: иначе рядом с новыми
    # цифрами остаются старые, и непонятно, каким верить
    loader.prune_sheets([name for name, _, _ in blocks])

    for name, df, group_col in blocks:
        if df is None or df.empty:
            log.warning("Вкладка «%s» пустая — пропускаю", name)
            continue

        table, breaks = _with_group_breaks(df, group_col) if group_col \
            else (df, [])
        table = table.rename(columns=RENAME_FOR_SHEET)
        ncol = len(table.columns)

        loader.open_sheet_by_name(name, rows=max(200, len(table) + 30),
                                  cols=max(20, ncol + 5))
        # Снимаем оформление прошлого запуска, иначе старые правила
        # лягут на новые колонки и штуки покажутся процентами
        loader.reset_sheet_format()
        loader.write([{"range": "A1", "values": Serializer.frame(table)}])

        text_tab = "Выводы" in name or "Справочник" in name
        reqs = [
            loader.req_header(0, 0, ncol, Palette.HEADER_DARK,
                              font_color=Palette.WHITE),
            loader.req_freeze(1),
            loader.req_borders(0, len(table) + 1, 0, ncol),
            loader.req_col_width(0, 1, 170),
        ]
        if text_tab:
            reqs.append(loader.req_col_width(1, ncol - 1, 150))
            reqs.append(loader.req_col_width(ncol - 1, ncol, 700))
        else:
            reqs.append(loader.req_col_width(1, ncol, 135))
            reqs.append(loader.req_center(1, len(table) + 1, 2, ncol))

        # Полосы-разделители между проектами
        for r in breaks:
            reqs.append(loader.req_row_fill(r, 0, ncol,
                                            Palette.HEADER_GREEN, bold=True))

        for j, col in enumerate(table.columns):
            src_col = next((k for k, v in RENAME_FOR_SHEET.items()
                            if v == col), col)
            if src_col in PERCENT_COLUMNS:
                reqs.append(loader.req_number_format(
                    1, len(table) + 1, j, j + 1, pattern="0.0%"))
            if src_col in WORSE_ON_GROWTH:
                reqs += loader.req_sign_colors(1, len(table) + 1, j, j + 1,
                                               growth_is_bad=True)
            elif src_col in BETTER_ON_GROWTH:
                reqs += loader.req_sign_colors(1, len(table) + 1, j, j + 1,
                                               growth_is_bad=False)
            if src_col == "Доля пропущенных":
                reqs.append(loader.req_color_scale(1, len(table) + 1, j, j + 1))
            if src_col in ("Уровень ответа", "Перезвонили", "SL АТС"):
                # Наоборот: чем выше, тем зеленее
                reqs.append(loader.req_color_scale(1, len(table) + 1, j, j + 1,
                                                   reverse=True))

        loader.apply_format(reqs, strict=False)
        log.info("Вкладка «%s»: %d строк, %d колонок", name, len(table), ncol)

    loader.reorder_sheets([name for name, _, _ in blocks])
    a, b = tables["период"]
    print(f"\nЗаписано: «{loader.sh.title}», период "
          f"{a.strftime('%d.%m')} — {b.strftime('%d.%m.%Y')}")


def upload_analytics() -> None:
    """Закрывает прошлый месяц, если пора, и выкладывает текущий."""
    loader = SheetsLoader(CREDENTIALS, ANALYTICS_ID).connect()

    # Сначала прошлый месяц: досчитать целиком, записать, сохранить в файл.
    # Каждая вкладка при записи очищается, так что старые данные уйдут
    # сами — отдельно стирать ничего не нужно.
    if ARCHIVE_PREVIOUS:
        close_previous_month(loader)

    write_tables(loader, TABLES)




upload_analytics()